# 02. Gaussian Filters — Kalman Filter와 EKF

Gaussian filter는 belief를 평균과 공분산으로 표현한다.

$$bel(x)=\mathcal{N}(\mu,\Sigma)$$

선형 Gaussian 시스템이면 Kalman Filter를 정확히 쓸 수 있고, 비선형이면 Jacobian으로 선형화한 EKF를 쓴다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import os
os.makedirs('assets', exist_ok=True)

for font_name in ['Nanum Gothic', 'AppleGothic', 'Malgun Gothic']:
    if any(font.name == font_name for font in fm.fontManager.ttflist):
        plt.rcParams['font.family'] = font_name
        break
plt.rcParams['axes.unicode_minus'] = False

## 1. Linear Gaussian System: Kalman Filter

상태 $x=[position, velocity]^T$, 측정은 위치만 들어온다.

In [ ]:
np.random.seed(2)
dt = 0.1
steps = 150
t = np.arange(steps)*dt
true_x = np.zeros((steps,2))
true_x[0] = [0, 1.0]
for k in range(steps-1):
    true_x[k+1,0] = true_x[k,0] + dt*true_x[k,1]
    true_x[k+1,1] = true_x[k,1] + 0.02*np.sin(0.6*t[k])
z = true_x[:,0] + np.random.randn(steps)*0.45

F = np.array([[1,dt],[0,1]])
H = np.array([[1,0]])
Q = np.diag([0.002,0.01])
R = np.array([[0.45**2]])
mu = np.array([0.0,0.0])
P = np.eye(2)*2
I = np.eye(2)
ests=[]; sig=[]
for k in range(steps):
    mu_bar = F @ mu
    P_bar = F @ P @ F.T + Q
    S = H @ P_bar @ H.T + R
    K = P_bar @ H.T @ np.linalg.inv(S)
    mu = mu_bar + (K @ (np.array([z[k]]) - H @ mu_bar)).ravel()
    P = (I - K @ H) @ P_bar
    ests.append(mu.copy()); sig.append(np.sqrt(np.diag(P)))
ests=np.array(ests); sig=np.array(sig)

fig, axes = plt.subplots(2,1,figsize=(10,7),sharex=True)
axes[0].plot(t,true_x[:,0],'k-',lw=2,label='true')
axes[0].scatter(t,z,s=12,color='gray',alpha=0.4,label='measurement')
axes[0].plot(t,ests[:,0],color='#E85D24',lw=2,label='KF')
axes[0].fill_between(t,ests[:,0]-2*sig[:,0],ests[:,0]+2*sig[:,0],color='#E85D24',alpha=0.15)
axes[0].legend(); axes[0].grid(alpha=0.25); axes[0].set_ylabel('position')
axes[1].plot(t,true_x[:,1],'k-',lw=2,label='true velocity')
axes[1].plot(t,ests[:,1],color='#534AB7',lw=2,label='estimated')
axes[1].legend(); axes[1].grid(alpha=0.25); axes[1].set_ylabel('velocity')
axes[1].set_xlabel('time')
plt.tight_layout(); plt.savefig('assets/02_kalman_filter.png',dpi=150,bbox_inches='tight'); plt.show()
print('position RMSE:', np.sqrt(np.mean((ests[:,0]-true_x[:,0])**2)).round(4))

## 2. EKF: 비선형 Range-Bearing 측정

랜드마크 위치 $m=[m_x,m_y]^T$가 주어졌을 때 로봇 pose $[x,y,\theta]$에서 측정은 다음과 같다.

$$h(x)=\begin{bmatrix}\sqrt{(m_x-x)^2+(m_y-y)^2}\\ atan2(m_y-y,m_x-x)-\theta\end{bmatrix}$$

EKF는 $H=\partial h/\partial x$로 관측 모델을 선형화한다.

In [ ]:
def wrap(a):
    return np.arctan2(np.sin(a), np.cos(a))

def h_range_bearing(x, landmark):
    dx, dy = landmark[0]-x[0], landmark[1]-x[1]
    return np.array([np.hypot(dx,dy), wrap(np.arctan2(dy,dx)-x[2])])

def H_jacobian(x, landmark):
    dx, dy = landmark[0]-x[0], landmark[1]-x[1]
    q = dx*dx + dy*dy
    r = np.sqrt(q)
    return np.array([[-dx/r, -dy/r, 0], [dy/q, -dx/q, -1]])

landmark = np.array([4.0, 2.0])
mu = np.array([0.5, -0.2, np.deg2rad(15)])
P = np.diag([0.4,0.25,np.deg2rad(12)**2])
true_pose = np.array([0.9, 0.15, np.deg2rad(20)])
z = h_range_bearing(true_pose, landmark) + np.array([0.08, np.deg2rad(-2)])
R = np.diag([0.12**2, np.deg2rad(4)**2])

z_hat = h_range_bearing(mu, landmark)
H = H_jacobian(mu, landmark)
S = H @ P @ H.T + R
K = P @ H.T @ np.linalg.inv(S)
innov = z - z_hat
innov[1] = wrap(innov[1])
mu_new = mu + K @ innov
mu_new[2] = wrap(mu_new[2])
P_new = (np.eye(3) - K @ H) @ P

print('before:', np.round([mu[0],mu[1],np.rad2deg(mu[2])],3))
print('after :', np.round([mu_new[0],mu_new[1],np.rad2deg(mu_new[2])],3))
print('true  :', np.round([true_pose[0],true_pose[1],np.rad2deg(true_pose[2])],3))
print('innovation:', np.round([innov[0], np.rad2deg(innov[1])],3))

fig, ax = plt.subplots(figsize=(7,6))
ax.scatter(*landmark, marker='*', s=180, color='#1D9E75', label='landmark')
ax.scatter(mu[0],mu[1],s=90,color='#E85D24',label='prior mean')
ax.scatter(mu_new[0],mu_new[1],s=90,color='#534AB7',label='posterior mean')
ax.scatter(true_pose[0],true_pose[1],s=90,color='black',label='true')
for p,c in [(mu,'#E85D24'),(mu_new,'#534AB7'),(true_pose,'black')]:
    ax.arrow(p[0],p[1],0.35*np.cos(p[2]),0.35*np.sin(p[2]),color=c,head_width=0.05,length_includes_head=True)
ax.set_aspect('equal'); ax.grid(alpha=0.25); ax.legend(); ax.set_title('EKF range-bearing update')
plt.savefig('assets/02_ekf_update.png',dpi=150,bbox_inches='tight'); plt.show()

## 요약

| 개념 | 의미 | 책 커리큘럼 연결 |
|------|------|------------------|
| Kalman Filter | 선형 Gaussian Bayes filter | Ch.3 Gaussian Filters |
| EKF | 비선형 모델을 Jacobian으로 선형화 | Ch.3 Extended Kalman Filter |
| 공분산 | 추정 불확실성 | Kalman gain과 센서 융합 가중치 |